# 👕 Fashion Image Classification — CNN Lab

**Dataset:** Fashion-MNIST (70,000 grayscale images of clothing)

**Goal:** Classify images into 10 categories using a Convolutional Neural Network.

| Label | Class | Label | Class |
|---|---|---|---|
| 0 | T-shirt/top | 5 | Sandal |
| 1 | Trouser | 6 | Shirt |
| 2 | Pullover | 7 | Sneaker |
| 3 | Dress | 8 | Bag |
| 4 | Coat | 9 | Ankle boot |

**What you'll learn:**
- How to load and preprocess image data
- How to build a CNN with Keras
- How to train, evaluate, and visualize results

---
## Step 1: Import Libraries

We import all the tools we need:
- **NumPy/Pandas** — data handling
- **Matplotlib/Seaborn** — visualization
- **TensorFlow/Keras** — build and train the CNN
- **Scikit-learn** — split data and evaluate metrics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_style('whitegrid')

import tensorflow as tf
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)

print(f"TensorFlow version: {tf.__version__}")

---
## Step 2: Load the Dataset

Fashion-MNIST comes built into Keras. We load:
- **60,000 training images** (28×28 pixels each)
- **10,000 test images**

Each image is a grayscale picture of a clothing item.

In [ ]:
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training images shape: {X_train_full.shape}")
print(f"Test images shape: {X_test.shape}")
print(f"Image size: {X_train_full[0].shape}")
print(f"Pixel range: {X_train_full.min()} to {X_train_full.max()}")

---
## Step 3: Visualize Sample Images

Let's look at 25 random images to understand what the data looks like.

In [ ]:
plt.figure(figsize=(10,10))
for i in range(25):
    plt.subplot(5, 5, i+1)
    plt.imshow(X_train_full[i], cmap='gray')
    plt.title(class_names[y_train_full[i]], fontsize=9)
    plt.axis('off')
plt.suptitle("Sample Fashion-MNIST Images", fontsize=14)
plt.tight_layout()
plt.show()

---
## Step 4: Preprocess the Data

Before feeding images to the CNN, we need to prepare them:

1. **Normalize** pixel values from `[0, 255]` → `[0, 1]` — helps training converge faster
2. **Reshape** to add channel dimension: `(28,28)` → `(28,28,1)` — CNN expects 4D input
3. **One-hot encode** labels: `3` → `[0,0,0,1,0,0,0,0,0,0]` — needed for categorical loss

In [ ]:
# Normalize pixels to [0, 1]
X_train_full = X_train_full.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Add channel dimension for CNN
X_train_full = X_train_full.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

# One-hot encode labels
y_train_full_cat = to_categorical(y_train_full, num_classes=10)
y_test_cat = to_categorical(y_test, num_classes=10)

print(f"X_train shape: {X_train_full.shape}")
print(f"y_train shape: {y_train_full_cat.shape}")

---
## Step 5: Split into Train / Validation / Test

We keep the test set untouched until the end. From training data, we carve out **10% for validation** to monitor overfitting during training.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full_cat, test_size=0.1, random_state=42, stratify=y_train_full
)

print(f"Training:   {X_train.shape[0]} images")
print(f"Validation: {X_val.shape[0]} images")
print(f"Test:       {X_test.shape[0]} images")

---
## Step 6: Build the CNN Model

Our CNN architecture:

| Layer | What it does |
|---|---|
| `Conv2D(32, 3x3)` + ReLU | Learns 32 feature filters (edges, corners) |
| `MaxPooling2D(2x2)` | Downsamples, keeps strongest features |
| `Conv2D(64, 3x3)` + ReLU | Learns 64 complex features |
| `MaxPooling2D(2x2)` | Downsamples again |
| `Flatten` | Converts 2D maps → 1D vector |
| `Dense(128)` + ReLU | Combines features |
| `Dropout(0.5)` | Prevents overfitting |
| `Dense(10)` + Softmax | Outputs probability for each class |

In [ ]:
model = Sequential([
    Conv2D(32, kernel_size=(3,3), activation='relu', input_shape=(28,28,1)),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(64, kernel_size=(3,3), activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(10, activation='softmax')
])

model.summary()

---
## Step 7: Compile & Train

- **Loss:** `categorical_crossentropy` — standard for multi-class
- **Optimizer:** `adam` — adaptive learning rate
- **Metric:** `accuracy`
- **Epochs:** 10 (full passes over training data)
- **Batch size:** 64 (images per gradient update)

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=64,
    verbose=1
)

---
## Step 8: Plot Training Curves

We visualize how accuracy and loss changed during training:
- **Good sign:** Both train and validation curves improve together
- **Bad sign:** Validation loss increases while training loss decreases = overfitting

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## Step 9: Evaluate on Test Set

We measure performance using:
- **Accuracy** — overall % correct
- **Precision** — of predicted positives, how many are correct?
- **Recall** — of actual positives, how many did we catch?
- **F1-Score** — harmonic mean of precision & recall
- **Confusion Matrix** — shows which classes get confused

In [ ]:
# Overall accuracy
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_accuracy:.4f}\n")

# Predictions
y_pred = np.argmax(model.predict(X_test), axis=1)

# Classification report (precision, recall, f1 per class)
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## Step 10: Visualize Predictions

Let's see the model's predictions on random test images:
- **Green** = correct prediction
- **Red** = incorrect prediction

In [ ]:
np.random.seed(1)
indices = np.random.choice(len(X_test), 15, replace=False)

plt.figure(figsize=(14, 8))
for i, idx in enumerate(indices):
    plt.subplot(3, 5, i+1)
    plt.imshow(X_test[idx].reshape(28,28), cmap='gray')
    pred_label = class_names[y_pred[idx]]
    true_label = class_names[y_test[idx]]
    color = 'green' if pred_label == true_label else 'red'
    plt.title(f"Pred: {pred_label}\nTrue: {true_label}", fontsize=9, color=color)
    plt.axis('off')

plt.suptitle("Predictions (green=correct, red=incorrect)", fontsize=14)
plt.tight_layout()
plt.show()

---
## Key Takeaways

1. **CNNs automatically learn visual features** — no need to hand-craft them
2. **Accuracy alone isn't enough** — check per-class precision/recall
3. **Dropout reduces overfitting** — randomly disables neurons during training
4. **Watch validation curves** — if val loss rises while train loss falls, model is overfitting

### Try these extensions:
- Add a third Conv2D block
- Add `BatchNormalization` layers
- Try data augmentation (`RandomFlip`, `RandomRotation`)
- Use `EarlyStopping` callback